In [13]:
from langchain_classic.chains.question_answering.map_reduce_prompt import messages

from config.llm import llm
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from langchain.tools import tool, ToolRuntime
from langchain.agents.middleware import dynamic_prompt, wrap_model_call, ModelRequest, ModelResponse, SummarizationMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from typing import Literal, Any

In [14]:
agent = create_agent(llm, system_prompt="you are a helpful assistant")
response = agent.invoke({'messages': '你好'})

response['messages'][-1].content

/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


'你好！有什么可以帮助你的吗？'

带工具的agent

In [2]:
def get_weather(city: str) -> str:
  """get weather for a given city"""
  return f"it's always sunny in {city}"

In [7]:
tool_agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[get_weather])

response = tool_agent.invoke({
  "messages": [{
    "role": "user",
    "content": "what is the weather in sf"
  }]
})

response['messages'][-1]

/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the 

AIMessage(content='\nI see that the weather function is returning a playful response about San Francisco being "always sunny." While San Francisco is known for its generally mild climate, it\'s actually not always sunny - the city experiences fog, especially in the summer months, and receives regular rainfall during the winter.\n\nIf you\'d like more specific weather information like current temperature, humidity, or detailed forecasts, you might want to check a dedicated weather service or app, as the function I have access to seems to be returning a simplified response.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 235, 'total_tokens': 338}, 'model_name': 'glm-4', 'finish_reason': 'stop'}, id='lc_run--019ca4b5-9014-76c3-8845-04b7752e26dd-0', tool_calls=[], invalid_tool_calls=[])

In [21]:
class Context(BaseModel):
    authority: Literal["admin", "user"]

class CalcInfo(BaseModel):
    """Calculation information."""
    output: int = Field(description="The calculation result")

@tool
def math_add(runtime: ToolRuntime[Context, Any], a: int, b: int) -> int:
    """Add two numbers together"""
    authority = runtime.context.authority
    if authority != "admin":
        raise PermissionError("user dose not have permission to add numbers")
    return a + b

agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[math_add, get_weather], )

# tool_agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[math_add, get_weather], response_format=CalcInfo)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "请计算 8234783 + 94123832 = ?"}]},
    config={"configurable": {"thread_id": "1"}},
    context=Context(authority="admin"))

for message in response['messages']:
    message.pretty_print()

/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


================================ Human Message =================================

请计算 8234783 + 94123832 = ?
================================== Ai Message ==================================


我来帮您计算这两个数字的和。
Tool Calls:
  math_add (call_-7848812936825921555)
 Call ID: call_-7848812936825921555
  Args:
    a: 8234783
    b: 94123832
================================= Tool Message =================================
Name: math_add

102358615
================================== Ai Message ==================================


8234783 + 94123832 = 102,358,615


In [3]:
agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[get_weather] )


for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'text', 'text': "\nI'll get the weather information for San Francisco (SF).\n"}, {'type': 'tool_call', 'id': 'call_-7848776824740899993', 'name': 'get_weather', 'args': {'city': 'SF'}}]
step: tools
content: [{'type': 'text', 'text': "it's always sunny in SF"}]
step: model
content: [{'type': 'text', 'text': '\nThe weather function returned a lighthearted response saying "it\'s always sunny in SF." This is a common phrase associated with San Francisco, though the actual weather can vary quite a bit throughout the year! \n\nSan Francisco typically has mild temperatures with cool, foggy mornings in the summer and occasional rain in the winter months. Would you like me to get more specific weather details for a particular date or time?'}]


In [11]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import ToolNode
from langchain_core.runnables import RunnableConfig

In [12]:
# 创建工具节点
tools = [get_weather]
tool_node = ToolNode(tools)

# 创建助手节点（修复核心错误）
def assistant(state: MessagesState, config: RunnableConfig):
    system_prompt = 'You are a helpful assistant that can check weather.'
    all_messages = [SystemMessage(system_prompt)] + state['messages']
    model = llm.bind_tools(tools=tools) 
    return {'messages': [model.invoke(all_messages)]}

# 创建条件边
def should_continue(state: MessagesState, config: RunnableConfig):
    messages = state['messages']
    last_message = messages[-1]
    if last_message.tool_calls:
        return 'continue'
    return 'end'

# 创建图
builder = StateGraph(MessagesState)

# 添加节点
builder.add_node('assistant', assistant)
builder.add_node('tool', tool_node)

# 添加边
builder.add_edge(START, 'assistant')

# 添加条件边
builder.add_conditional_edges(
    'assistant',
    should_continue,
    {
        'continue': 'tool',
        'end': END,
    },
)

# 添加边：调用工具节点后回到assistant
builder.add_edge('tool', 'assistant')

# 编译图
my_graph = builder.compile(name='my-graph')

# 调用图并输出结果
response = my_graph.invoke({'messages': [HumanMessage(content='上海天气怎么样？')]})
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

上海天气怎么样？
================================== Ai Message ==================================


我来帮您查询上海的天气情况。
Tool Calls:
  get_weather (call_-7848641481731472376)
 Call ID: call_-7848641481731472376
  Args:
    city: 上海
================================= Tool Message =================================
Name: get_weather

It's always sunny in 上海!
================================== Ai Message ==================================


根据查询结果，上海目前天气晴朗！阳光明媚，是个好天气。如果您需要更详细的天气信息（如温度、湿度、风力等），请告诉我，我可以为您进一步查询。


In [17]:
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers and return the sum."""
    return a + b

In [18]:
@tool
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate BMI given weight in kg and height in meters."""
    if height_m <= 0 or weight_kg <= 0:
        raise ValueError("height_m and weight_kg must be greater than 0.")
    return weight_kg / (height_m ** 2)

In [19]:
tool_agent = create_agent(
    model=llm,
    tools=[get_weather, add_numbers, calculate_bmi],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # 无需触发人工审批
                "get_weather": False,
                # 需要审批，且允许approve,edit,reject三种审批类型
                "add_numbers": True,
                # 需要审批，允许approve,reject两种审批类型
                "calculate_bmi": {"allowed_decisions": ["approve", "reject"]},
            },
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),
    system_prompt="You are a helpful assistant",
)

In [20]:
import uuid
config = {'configurable': {'thread_id': str(uuid.uuid4())}}
result = tool_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "我身高180cm，体重180斤，我的BMI是多少"
        # "content": "what is the weather in sf"
    }]},
    config=config,
)

# result['messages'][-1].content
result.get('__interrupt__')

/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


[Interrupt(value={'action_requests': [{'name': 'calculate_bmi', 'args': {'weight_kg': 90, 'height_m': 1.8}, 'description': "Tool execution pending approval\n\nTool: calculate_bmi\nArgs: {'weight_kg': 90, 'height_m': 1.8}"}], 'review_configs': [{'action_name': 'calculate_bmi', 'allowed_decisions': ['approve', 'reject']}]}, id='5cd8c6dfebe1b1b130874fba2709fee5')]

In [21]:
from langgraph.types import Command
result = tool_agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # or "edit", "reject"
    ),
    config=config
)

result['messages'][-1].content

/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


'\n根据您提供的信息：\n- 身高：180cm = 1.8米\n- 体重：180斤 = 90公斤\n\n您的BMI是 **27.78**\n\n这个BMI值属于"超重"范围（25-29.9）。建议您：\n- 保持均衡饮食\n- 适量运动\n- 定期监测体重变化\n\n如果您需要更详细的健康建议，建议咨询专业医生。'

In [13]:
@dynamic_prompt
def state_aware_prompt(request: ModelRequest) -> str:
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)

    base = "You are a helpful assistant."

    if message_count > 6:
        base += "\nThis is a long conversation - be extra concise."

    # 临时打印base看效果
    print(base)

    return base

agent = create_agent(
    model=llm,
    middleware=[state_aware_prompt]
)

result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "广州今天的天气怎么样？"},
        {"role": "assistant", "content": "广州天气很好"},
        {"role": "user", "content": "吃点什么好呢"},
        {"role": "assistant", "content": "要不要吃香茅鳗鱼煲"},
        {"role": "user", "content": "香茅是什么"},
        {"role": "assistant", "content": "香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉"},
        {"role": "user", "content": "auv 那还等什么，咱吃去吧"},
    ]},
)

for message in result['messages']:
    message.pretty_print()

You are a helpful assistant.
This is a long conversation - be extra concise.
================================ Human Message =================================

广州今天的天气怎么样？
================================== Ai Message ==================================

广州天气很好
================================ Human Message =================================

吃点什么好呢
================================== Ai Message ==================================

要不要吃香茅鳗鱼煲
================================ Human Message =================================

香茅是什么
================================== Ai Message ==================================

香茅又名柠檬草，常见于泰式冬阴功汤、越南烤肉
================================ Human Message =================================

auv 那还等什么，咱吃去吧
================================== Ai Message ==================================


走！就去尝尝香茅鳗鱼煲


In [23]:
import os
import uuid
import sqlite3
from langgraph.store.sqlite import SqliteStore

In [24]:
# 删除SQLite数据库
if os.path.exists("user-info.db"):
    os.remove("user-info.db")

# 创建SQLite存储
conn = sqlite3.connect("user-info.db", check_same_thread=False, isolation_level=None)
conn.execute("PRAGMA journal_mode=WAL;")
conn.execute("PRAGMA busy_timeout = 30000;")

store = SqliteStore(conn)

# 预置两条用户信息
store.put(("user_info",), "柳如烟", {"description": "清冷才女，身怀绝技，为寻身世之谜踏入江湖。", "birthplace": "吴兴县"})
store.put(("user_info",), "苏慕白", {"description": "孤傲剑客，剑法超群，背负家族血仇，隐于市井追寻真相。", "birthplace": "杭县"})

In [25]:
@tool
def fetch_user_data(
    user_id: str,
    runtime: ToolRuntime
) -> str:
    """
    Fetch user information from the in-memory store.

    :param user_id: The unique identifier of the user.
    :param runtime: The tool runtime context injected by the framework.
    :return: The user's description string if found; an empty string otherwise.
    """
    store = runtime.store
    user_info = store.get(("user_info",), user_id)

    user_desc = ""
    if user_info:
        user_desc = user_info.value.get("description", "")

    return user_desc

In [26]:
agent = create_agent(
    model=llm,
    tools=[fetch_user_data],
    store=store,
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "五分钟之内，我要柳如烟的全部信息"
    }]
})

for message in result['messages']:
    message.pretty_print()

/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


================================ Human Message =================================

五分钟之内，我要柳如烟的全部信息
================================== Ai Message ==================================


我来帮您获取柳如烟的全部信息。
Tool Calls:
  fetch_user_data (call_-7848538196357934122)
 Call ID: call_-7848538196357934122
  Args:
    user_id: 柳如烟
================================= Tool Message =================================
Name: fetch_user_data

清冷才女，身怀绝技，为寻身世之谜踏入江湖。
================================== Ai Message ==================================


我已经为您获取到了柳如烟的信息：

**柳如烟的全部信息：**

身份：清冷才女
特点：身怀绝技
背景：为寻身世之谜踏入江湖

柳如烟是一位性格清冷的才女，拥有非凡的技艺。她为了解开自己身世的谜团，选择踏入江湖，开始了她的寻亲之旅。


In [36]:
from langchain_mcp_adapters.client import MultiServerMCPClient

async def mcp_agent():
    # 我们用两种方式启动 MCP Server：stdio 和 streamable_http
    client = MultiServerMCPClient(  
        {
            "weather": {
                "url": "http://localhost:8001/mcp",
                "transport": "streamable_http",
            }
        }
    )
    
    tools = await client.get_tools()
    agent = create_agent(
        llm,
        tools=tools,
    )

    return agent

async def use_mcp(messages):
    agent = await mcp_agent()
    response = await agent.ainvoke(messages)
    return response

messages = {"messages": [{"role": "user", "content": "福州天气怎么样？"}]}
response = await use_mcp(messages)
response["messages"][-1].content
   

/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


'\n根据查询结果，福州目前的天气状况是：**阳光明媚**！\n\n福州的天气看起来很不错，是个阳光灿烂的好天气。如果您需要了解其他城市的天气情况，我也可以为您查询。'

In [37]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide two numbers."""
    return a / b

In [38]:
subagent1 = create_agent(llm, tools=[add], name="subagent1")

@tool("subagent1", description="可以准确地计算两数想加")
def call_subagent1(query: str):
  result = subagent1.invoke({
            "messages": [{"role": "user", "content": query}]
  })
  
  return result['messages'][-1].content

# 创建subagent2：用于计算两数相乘
subagent2 = create_agent(
    model=llm,
    tools=[multiply],
    name="subagent-2",
)

@tool(
    "subagent-2",
    description="可以准确地计算两数相乘"
)
def call_subagent2(query: str):
    result = subagent2.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].content

# 创建supervisor agent
supervisor_agent = create_agent(
    model=llm,
    tools=[call_subagent1, call_subagent2, divide],
    name="supervisor-agent",
    system_prompt="提示：如遇两数相减仍可用两数相加工具实现，只需将一个数加上另一个数的负数",
)


result = supervisor_agent.invoke({
    "messages": [{"role": "user", "content": "计算 38462 + 378 / 49 * 83723 - 123 的结果"}]}
)

for message in result["messages"]:
    message.pretty_print()

/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 

================================ Human Message =================================

计算 38462 + 378 / 49 * 83723 - 123 的结果
================================== Ai Message ==================================
Name: supervisor-agent


我来帮你计算这个表达式。这个表达式包含多个运算，我需要按照运算顺序来计算。让我一步一步来：

首先，我需要计算除法：378 / 49
Tool Calls:
  divide (call_-7848528163314333022)
 Call ID: call_-7848528163314333022
  Args:
    a: 378
    b: 49
================================= Tool Message =================================
Name: divide

7.714285714285714
================================== Ai Message ==================================
Name: supervisor-agent
Tool Calls:
  subagent-2 (call_-7848525689413170538)
 Call ID: call_-7848525689413170538
  Args:
    query: 7.714285714285714 * 83723
================================= Tool Message =================================
Name: subagent-2


The result of multiplying 7.714285714285714 by 83723 is **645,863.1428571428**.
================================== Ai Message ================

In [39]:
from langgraph_supervisor import create_supervisor

subagent3 = create_agent(
    model=llm,
    tools=[divide],
    name="subagent-3",
)


supervisor_graph = create_supervisor(
    [subagent1, subagent2, subagent3],
    model=llm,
    prompt="提示：如遇两数相减仍可用两数相加工具实现，只需将一个数加上另一个数的负数"
)

supervisor_app = supervisor_graph.compile()

result = supervisor_app.invoke({
    "messages": [{"role": "user", "content": "计算 38462 + 378 / 49 * 83723 - 123 的结果"}]}
)

for message in result["messages"]:
    message.pretty_print()

/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 

================================ Human Message =================================

计算 38462 + 378 / 49 * 83723 - 123 的结果
================================== Ai Message ==================================
Name: supervisor


我来帮你计算这个数学表达式。让我逐步计算：
Tool Calls:
  transfer_to_subagent1 (call_-7848511876798341369)
 Call ID: call_-7848511876798341369
  Args:
================================= Tool Message =================================
Name: transfer_to_subagent1

Successfully transferred to subagent1
================================== Ai Message ==================================
Name: subagent1


根据数学运算顺序（先乘除后加减），让我重新计算这个表达式：

**步骤 1：计算除法**
378 ÷ 49 = 7.714285714285714

**步骤 2：计算乘法**
7.714285714285714 × 83723 = 646,028.5714285714

**步骤 3：进行加减运算**
38462 + 646,028.5714285714 - 123 = 684,367.5714285714

**最终结果：**
38462 + 378 ÷ 49 × 83723 - 123 = **684,367.57**

注意：由于除法结果不是整数，所以最终结果是一个小数。
================================== Ai Message ==================================
Name: subagent1

Transferrin

In [40]:
from langgraph.func import entrypoint, task

# 可并发的执行单元
@task
def generate_paragraph(topic: str) -> str:
  response = llm.invoke([
    {"role": "system", "content": "你是一个会写科普文的智能助手"},
    {"role": "user", "content": (
            f"写一个关于{topic}的段落，大约100字。"
            "你写的内容只是整篇文章的一个段落"
            "因此请聚焦给定话题，不需要概述性文字")}
  ])
  
  return response.content

# 创建持久化检查点
checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def workflow(topics: list[str]) -> str:
   futures = [generate_paragraph(topic) for topic in topics]
   paragraphs = [f.result() for f in futures]
   return "\n\n".join(paragraphs)


config = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = workflow.invoke(["海胆豆腐是什么", "海胆豆腐有多美味", "海胆豆腐的原料和做法"], config=config)
print(result)

海胆豆腐并非传统意义上的豆腐，而是由海胆的生殖腺与豆腐或淀粉混合制成的日式料理。它保留了海胆独特的鲜甜口感和金黄色泽，质地如丝般顺滑细腻。这种创新料理既保留了海胆的原始风味，又通过豆腐的加入赋予其新的口感层次，成为日式料理中备受推崇的特色美食。

海胆豆腐入口即化，如海洋的精华在舌尖绽放。金黄色的海胆与嫩滑的豆腐完美融合，咸鲜中带着淡淡的甜意，细腻的口感如同云朵般轻盈。每一口都充满海洋的鲜美，余韵悠长，让人回味无穷，仿佛置身于蔚蓝深海之中，品尝着大自然最纯粹的馈赠。

海胆豆腐主要原料为海胆和豆腐。选用新鲜海胆取出卵黄，与嫩豆腐按比例混合，加入少许盐和糖调味。将混合物轻轻拌匀，保持海胆颗粒感，最后冷藏定型即可。成品色泽金黄，口感滑嫩，海胆的鲜甜与豆腐的清香完美融合，是一道极具海洋风味的精致料理。
